# AI-Enhanced Admission Management System
## Machine Learning Module: Admission Eligibility Prediction
**Algorithm:** Logistic Regression (Scikit-Learn)
**Objective:** Build an end-to-end classification pipeline that predicts student admission eligibility along with a transparent probability score based on academic and demographic profiles.

### 1. Import Required Libraries and Setup Environment

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print('Environment initialized successfully.')

### 2. Load Admission Dataset
The dataset contains educational admission applications with the required features:
- `academic_gpa`: Academic marks / GPA percentage (50-100)
- `entrance_score`: Entrance examination score (0-100)
- `attendance_pct`: Past attendance percentage
- `prev_academic_score`: 10th / 12th board percentage
- `category`: Reservation / category quota (General, OBC, SC_ST, EWS)
- `extracurricular`: Sports, leadership, and olympiad achievements
- `admission_status`: 1 = Eligible for Admission, 0 = Not Eligible

In [2]:
df = pd.read_csv('admission_dataset.csv')
print(f'Total Applications in Dataset: {len(df)}')
df.head(10)

### 3. Exploratory Data Analysis & Statistics

In [3]:
print('--- Dataset Summary Statistics ---')
print(df.describe())
print('\n--- Target Class Distribution ---')
print(df['admission_status'].value_counts(normalize=True))
print('\n--- Category Distribution ---')
print(df['category'].value_counts())
print('\n--- Extracurricular Distribution ---')
print(df['extracurricular'].value_counts())

### 4. Data Preprocessing & Pipeline Construction
We construct a production-ready `ColumnTransformer`:
1. Continuous numerical attributes normalized using `StandardScaler`
2. Categorical features encoded with `OneHotEncoder(drop='first')`
3. Integrated into a unified Scikit-learn `Pipeline` with `LogisticRegression`

In [4]:
X = df[['academic_gpa', 'entrance_score', 'attendance_pct', 'prev_academic_score', 'category', 'extracurricular']]
y = df['admission_status']

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_cols = ['academic_gpa', 'entrance_score', 'attendance_pct', 'prev_academic_score']
cat_cols = ['category', 'extracurricular']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42))
])

print('Pipeline configured with StandardScaler, OneHotEncoder, and LogisticRegression.')

### 5. Cross-Validation and Model Training

In [5]:
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
print(f'5-Fold Cross-Validation Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

# Fit model on entire training set
pipeline.fit(X_train, y_train)
print('Logistic Regression Model training completed.')

### 6. Model Evaluation on Test Set
Calculating comprehensive classification metrics:
- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC Score
- Confusion Matrix

In [6]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print('================ MODEL EVALUATION METRICS ================')
print(f'Accuracy : {acc * 100:.2f}%')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1-Score : {f1:.4f}')
print(f'ROC-AUC  : {roc:.4f}')
print('\nConfusion Matrix:')
print(cm)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Eligible', 'Eligible']))

### 7. Feature Importance & Logistic Regression Coefficients
Inspecting the learned feature weights to understand decision transparency.

In [7]:
ohe_feature_names = pipeline.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(cat_cols)
all_features = numeric_cols + list(ohe_feature_names)
coefficients = pipeline.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({
    'Feature': all_features,
    'Coefficient': np.round(coefficients, 4),
    'Odds_Ratio': np.round(np.exp(coefficients), 4)
}).sort_values(by='Coefficient', ascending=False)

print('--- Feature Coefficients & Odds Ratios ---')
print(coef_df.to_string(index=False))

### 8. Test Sample Inference (Eligibility & Probability Score)

In [8]:
# Test with high-performing applicant
sample_high = pd.DataFrame([{
    'academic_gpa': 88.5,
    'entrance_score': 91.0,
    'attendance_pct': 92.0,
    'prev_academic_score': 87.0,
    'category': 'General',
    'extracurricular': 'State_Level'
}])

prob_high = pipeline.predict_proba(sample_high)[0][1]
pred_high = pipeline.predict(sample_high)[0]
print(f'Sample 1 (High Merit): Predicted Status = {"Eligible" if pred_high==1 else "Not Eligible"}, Probability = {prob_high*100:.2f}%')

# Test with borderline applicant
sample_low = pd.DataFrame([{
    'academic_gpa': 54.0,
    'entrance_score': 42.0,
    'attendance_pct': 60.0,
    'prev_academic_score': 52.0,
    'category': 'General',
    'extracurricular': 'None'
}])

prob_low = pipeline.predict_proba(sample_low)[0][1]
pred_low = pipeline.predict(sample_low)[0]
print(f'Sample 2 (Low Merit) : Predicted Status = {"Eligible" if pred_low==1 else "Not Eligible"}, Probability = {prob_low*100:.2f}%')

### 9. Export Trained Model Artifact
Saving the trained pipeline for backend API integration.

In [9]:
joblib.dump(pipeline, 'admission_logistic_model.joblib')
print('Model successfully saved to admission_logistic_model.joblib for production integration.')